In [ ]:
!nvidia-smi

In [ ]:
import sys
# sys.path.append('./src')

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

# NEW gigalens "scene" API
from gigalens.jax.inference import ModellingSequence
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.scene_prob_model import Dataset, ProbModel
from gigalens.jax.scene_simulator import SceneSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

import tensorflow_probability.substrates.jax as tfp
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
import matplotlib as mpl
from matplotlib import pyplot as plt
tfd = tfp.distributions

In [ ]:
# Per-component scene priors. Same distributions as the original (old-API) notebook,
# now expressed as one prior dict per profile (the scene API derives the joint prior +
# bijector from these). Forward mode => the Sersic Ie amplitudes are SAMPLED params.
epl_p = dict(
    theta_E=tfd.LogNormal(jnp.log(1.25), 0.25),
    gamma=tfd.TruncatedNormal(2, 0.25, 1, 3),
    e1=tfd.Normal(0, 0.1),
    e2=tfd.Normal(0, 0.1),
    center_x=tfd.Normal(0, 0.05),
    center_y=tfd.Normal(0, 0.05),
)
shear_p = dict(
    gamma1=tfd.Normal(0, 0.05),
    gamma2=tfd.Normal(0, 0.05),
)
lens_light_p = dict(
    R_sersic=tfd.LogNormal(jnp.log(1.0), 0.15),
    n_sersic=tfd.Uniform(2, 6),
    e1=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
    e2=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
    center_x=tfd.Normal(0, 0.05),
    center_y=tfd.Normal(0, 0.05),
    Ie=tfd.LogNormal(jnp.log(500.0), 0.3),
)
source_light_p = dict(
    R_sersic=tfd.LogNormal(jnp.log(0.25), 0.15),
    n_sersic=tfd.Uniform(0.5, 4),
    e1=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
    e2=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
    center_x=tfd.Normal(0, 0.25),
    center_y=tfd.Normal(0, 0.25),
    Ie=tfd.LogNormal(jnp.log(150.0), 0.5),
)

Load the data. The ground truth parameters are in `truth`. Observation parameters, including the noise scale and exposure time are fixed.

The PSF is generated from `TinyTim` for HST F140W band, and has been supersampled to the pixel scale of 0.065.

In [ ]:
truth = [[
    {'theta_E': 1.1, 'gamma': 2.0, 'e1': 0.1, 'e2': 0.1, 'center_y': 0.0, 'center_x': 0.1},
    {'gamma2': 0.03, 'gamma1': -0.01}
], [
    {'R_sersic': 0.8, 'n_sersic': 2.5, 'e1': 0.09534746574143645, 'e2': 0.14849487967198177, 'center_x': 0.1, 'center_y': 0.0, 'Ie': 499.3695906504067}
], [
    {'R_sersic': 0.25, 'n_sersic': 1.5, 'e1': 0., 'e2': 0., 'center_x': 0.09566681002252231, 'center_y': -0.0639623054267272, 'Ie': 149.58828877085668}
]]

In [ ]:
# Asset paths: the bundled HST PSF + demo image live in the gigalens package.
ASSETS = "/global/u1/l/linusu/gigalens/src/gigalens/assets"
kernel = np.load(f"{ASSETS}/psf.npy").astype(np.float32)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=2, kernel=kernel)

# Scene LensModel: plane 0 = primary deflector (EPL + Shear mass) + Sersic lens light;
# plane 1 = lensed source (Sersic). use_lstsq=False => Ie amplitudes are sampled (dim=22).
lens_light = Component(sersic.SersicEllipse(use_lstsq=False), lens_light_p)
source_light = Component(sersic.SersicEllipse(use_lstsq=False), source_light_p)
model = LensModel([
    Plane(mass=[Component(epl.EPL(50), epl_p), Component(shear.Shear(), shear_p)],
          light=[lens_light]),
    Plane(deflection_ratio=1.0, light=[source_light]),
])

observed_img = np.load(f"{ASSETS}/demo.npy")
ds = Dataset(jnp.asarray(observed_img), sim_config,
             background_rms=0.2, exp_time=100, sees="all")
prob_model = ProbModel(model, ds, mode="forward")   # forward = old ForwardProbModel
model_seq = ModellingSequence(prob_model)
print("dim (num free params):", model.num_free_params)

In [ ]:
plt.imshow(observed_img, vmin=0, vmax=10)
plt.colorbar()

Sanity check: calculate residuals using ground truth.

In [ ]:
# Render the ground-truth scene with the new SceneSimulator and check residuals.
# Convert the old nested-list truth into the scene's unique-key -> value dict, then
# scatter it to the structured params dict the simulator consumes.
truth_unique = {}
for pname, v in truth[0][0].items(): truth_unique[f"planes/0/mass/0/{pname}"] = jnp.asarray(float(v))
for pname, v in truth[0][1].items(): truth_unique[f"planes/0/mass/1/{pname}"] = jnp.asarray(float(v))
for pname, v in truth[1][0].items(): truth_unique[f"planes/0/light/0/{pname}"] = jnp.asarray(float(v))
for pname, v in truth[2][0].items(): truth_unique[f"planes/1/light/0/{pname}"] = jnp.asarray(float(v))
truth_params = model.to_params(truth_unique)
simulated = np.asarray(prob_model.simulators[0].simulate(truth_params))

plt.figure(figsize=(8, 3))
plt.subplot(121)
plt.imshow(simulated, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=20))
plt.colorbar()
plt.axis('off')
plt.subplot(122)
resid = simulated - observed_img
background_rms, exp_time = 0.2, 100
err_map = np.sqrt(background_rms**2 + simulated/exp_time)
plt.imshow(resid/err_map, cmap='coolwarm', interpolation='none', vmin=-5, vmax=5)
print('Chi-square:', np.mean((resid/err_map)**2))
plt.axis('off')
plt.colorbar()

Begin fitting. We use `supersample=1` for this demonstration to speed things up, but setting `supersample=2` is recommended in practice. This will not significantly slow down the fitting.

In [ ]:
opt = optax.adabelief(1e-2, b1=0.95, b2=0.99)
# Scene-API MAP returns (best_z, best_logprob, chisq_history); best_z is the single
# best unconstrained position (no manual reselection needed -- the old map_estimate bug).
best, best_lp, best_chisq = model_seq.MAP(opt, seed=0)

In [ ]:
opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
qz, loss_hist = model_seq.SVI(best, opt, n_vi=1000, num_steps=1500)

In [ ]:
plt.plot(loss_hist)

Although the loss is still declining, the VI results are sufficient for our sampling.

In [ ]:
# LAPS handoff (replaces HMC). Two-phase Late-Adjusted Parallel Sampler; the
# validated defaults are baked into run_laps. init_mode="warm" seeds the chains from
# the SVI surrogate qz. res.samples is (num_chains, 22) in UNCONSTRAINED space
# (one sample per chain = the posterior ensemble).
import sys
sys.path.insert(0, "/global/u1/l/linusu/GIGALens-Code/experiments/laps_validation")
sys.path.insert(0, "/global/u1/l/linusu/GIGALens-Code/src")   # for gigalens_research
from handoff.laps_handoff import run_laps, diagnose   # compare_warm_cold also available

res = run_laps(model_seq, qz, init_mode="warm", num_chains=512)
health = diagnose(res)   # ground-truth-free internal-health report + plots -> laps_diagnose.png

# A second, independent cold init that AGREES is the key ground-truth-free convergence
# evidence on a real posterior (no known moments). It is available via:
# from handoff.laps_handoff import compare_warm_cold; compare_warm_cold(model_seq, qz)

In [ ]:
# LAPS returns ONE sample per chain (res.samples is (num_chains, 22)), so there is no
# within-chain time series for a classical autocorrelation R-hat / ESS. diagnose() already
# computed a between-subensemble split-R-hat + cross-chain ESS on the final ensemble:
print(f"split-Rhat max      : {health['rhat_max']:.4f}   (flag if >= 1.01)")
print(f"cross-chain ESS min : {health['ess_min']:.0f} / M={health['M']}")

In [ ]:
# Convert the LAPS ensemble (unconstrained z) to physical space via the scene bijector.
smp = res.samples.reshape((-1, 22))            # (num_chains, 22), UNCONSTRAINED
phys = prob_model.bij.forward(list(smp.T))     # dict: unique-key -> constrained (N,) array

# The 8 lens-mass params are EPL (theta_E, gamma, e1, e2, center_y, center_x) + external
# Shear (gamma2, gamma1). Order the columns to match the truth markers + labels below.
mass_order = [
    "planes/0/mass/0/theta_E",
    "planes/0/mass/0/gamma",
    "planes/0/mass/0/e1",
    "planes/0/mass/0/e2",
    "planes/0/mass/0/center_y",
    "planes/0/mass/0/center_x",
    "planes/0/mass/1/gamma2",
    "planes/0/mass/1/gamma1",
]
mass_samps = np.stack([np.asarray(phys[k]) for k in mass_order], axis=1)   # (N, 8)
mass_labels = mass_order

In [ ]:
markers = []
for i in truth[0][0].keys():
    markers.append(truth[0][0][i])
for j in truth[0][1].keys():
    markers.append(truth[0][1][j])

In [ ]:
from corner import corner
plt.style.use('default')

labels=[r'$\theta_E$', 
        r'$\gamma$', 
        r'$\epsilon_2$', 
        r'$\epsilon_1$',
        r'$y$', r'$x$', 
        r'$\gamma_{2,ext}$',
        r'$\gamma_{1,ext}$',]

fig = corner(mass_samps, show_titles=True, title_fmt='.3f', labels=labels, truths=markers)
fig.suptitle("Lens Parameters")
fig.savefig("laps_corner.png", dpi=110, bbox_inches="tight")
fig   # emit the corner figure inline